In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import os
import matplotlib.pyplot as plt
import json

In [ ]:
base_folder = r"/storage/alplakes_test/lucerne_100m_2025"

In [ ]:
wind_folder = os.path.join(base_folder, 'wind_analysis')

In [ ]:
ke_folder = os.path.join(base_folder, 'outputs_swirl', 'ke_eddy')

In [ ]:
eddy_stats_folder = os.path.join(base_folder, "outputs_swirl", "eddy_statistics")

# Import datasets

In [ ]:
ds_wind = xr.open_dataset(os.path.join(wind_folder, "wind_stats_per_zone.nc"))

In [ ]:
df_eddy_ke = pd.read_csv(os.path.join(ke_folder, "ke_eddies_-0.25--9.32m.csv"), index_col=0)
df_lake_ke = pd.read_csv(os.path.join(ke_folder, "ke_lake_-0.25--9.32m.csv"), index_col=0)

In [ ]:
df_eddy_ke['date'] = pd.to_datetime(df_eddy_ke['date'])
df_lake_ke['date'] = pd.to_datetime(df_lake_ke['date'])

In [ ]:
df_eddy_ke = df_eddy_ke.set_index('date')
df_lake_ke = df_lake_ke.set_index('date')

# Plot wind and KE

In [ ]:
fig, ax = plt.subplots()
df_lake_ke.diff().plot(ax=ax)
df_eddy_ke.diff().plot(ax=ax)
plt.ylim(-200,200)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

df_eddy_ke.plot(ax=ax1, c='darkorange', zorder=3)
df_lake_ke.plot(ax=ax1, c='blue', zorder=3)

ax2 = ax1.twinx()
ds_wind.sel(zone='all').speed.plot(ax=ax2, c='green', zorder=1)

# Put wind axis behind the main axis (so its artists stay visually behind)
ax1.set_zorder(1)
ax1.set_axisbelow(True)
ax1.patch.set_alpha(0)  # transparent background so wind is visible

# --- Grid control ---
ax1.grid(True)           # keep grid for KE axis (optional)
ax2.grid(False)          # remove grid for wind axis

ax1.set_ylabel('Kinetic Energy (MJ)')
ax2.set_ylabel('Wind Speed (m/s)')

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

# --- LEFT axis: wind ---
ds_wind.sel(zone="all").speed.plot(ax=ax1, c="green", zorder=1, add_legend=False, alpha=0.5)
ax1.set_ylabel("Wind Speed (m/s)")
ax1.set_title("")

# --- RIGHT axis 1: lake KE ---
ax2 = ax1.twinx()
df_lake_ke.plot(ax=ax2, c="blue", zorder=5, legend=False)
ax2.set_ylabel("Lake KE (MJ)")

# --- RIGHT axis 2 (shifted outward): eddy KE ---
ax3 = ax1.twinx()
ax3.spines["right"].set_position(("outward", 60))
df_eddy_ke.plot(ax=ax3, c="darkorange", zorder=6, legend=False)
ax3.set_ylabel("Eddy KE (MJ)")

# --- grid / layering ---
ax1.set_axisbelow(True)
ax1.grid(True, zorder=0)

ax2.grid(False)
ax3.grid(False)

ax1.tick_params(axis="y", colors="green")
ax1.yaxis.label.set_color("green")

ax2.tick_params(axis="y", colors="blue")
ax2.yaxis.label.set_color("blue")

ax3.tick_params(axis="y", colors="darkorange")
ax3.yaxis.label.set_color("darkorange")

# --- x limits ---
month = 9
xlim = (f"2025-{month:02}-01 00:30:00", f"2025-{month+2:02}-01 00:30:00")
ax2.set_xlim(xlim)

plt.text(0.02, 0.98, '-0.25 to -9.32m', transform=plt.gca().transAxes, ha='left', va='top')

fig.tight_layout()
fig.savefig(os.path.join(ke_folder, f'wind_eddy_analysis_-0.25--9.32m_{xlim[0].replace(":", "-")}_{xlim[1].replace(":", "-")}.png'))

# Plot wind and # of eddies

In [ ]:
nb_eddy = pd.read_csv(os.path.join(eddy_stats_folder, f"eddy_numbers_-10-0m.csv"), index_col=0)
nb_eddy['date'] = pd.to_datetime(nb_eddy['date'])
nb_eddy = nb_eddy.set_index('date')

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

# --- LEFT axis: wind ---
ds_wind.sel(zone="all").speed.plot(ax=ax1, c="green", zorder=1, add_legend=False, alpha=0.5)
ax1.set_ylabel("Wind Speed (m/s)")
ax1.set_title("")

# --- RIGHT axis 1: lake KE ---
ax2 = ax1.twinx()
df_lake_ke.plot(ax=ax2, c="blue", zorder=5, legend=False)
ax2.set_ylabel("Lake KE (MJ)")

# --- RIGHT axis 2 (shifted outward): eddy KE ---
ax3 = ax1.twinx()
ax3.spines["right"].set_position(("outward", 60))
df_eddy_ke.plot(ax=ax3, c="darkorange", zorder=6, legend=False)
ax3.set_ylabel("Eddy KE (MJ)")

# --- RIGHT axis 2 (shifted outward): eddy KE ---
ax4 = ax1.twinx()
ax4.spines["right"].set_position(("outward", 120))
nb_eddy.plot(ax=ax4, c="black", zorder=6, legend=False, alpha=0.6)
ax4.set_ylabel("Number of Eddy [-]")

# --- grid / layering ---
ax1.set_axisbelow(True)
ax1.grid(True, zorder=0)

ax2.grid(False)
ax3.grid(False)

ax1.tick_params(axis="y", colors="green")
ax1.yaxis.label.set_color("green")

ax2.tick_params(axis="y", colors="blue")
ax2.yaxis.label.set_color("blue")

ax3.tick_params(axis="y", colors="darkorange")
ax3.yaxis.label.set_color("darkorange")

ax4.tick_params(axis="y", colors="black")
ax4.yaxis.label.set_color("black")

# --- x limits ---
month = 9
xlim = (f"2025-{month:02}-01 00:30:00", f"2025-{month+2:02}-01 00:30:00")
ax2.set_xlim(xlim)

plt.text(0.02, 0.98, '-0.25 to -9.32m', transform=plt.gca().transAxes, ha='left', va='top')

fig.tight_layout()
fig.savefig(os.path.join(ke_folder, f'wind_eddy_number_analysis_-0.25--9.32m_{xlim[0].replace(":", "-")}_{xlim[1].replace(":", "-")}.png'))